In [ ]:
!pip install gradio faster-whisper requests
!apt-get install zstd -y
!curl -fsSL https://ollama.com/install.sh | sh

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 90.7 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (556 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8

In [ ]:
import subprocess
import time

# 1. Start Ollama in the background
print("Starting Ollama server...")
process = subprocess.Popen(["ollama", "serve"])
time.sleep(3) # Give it a moment to boot

# 2. Pull the Llama 3.2 model
print("Pulling Llama 3.2 model... (This takes a minute or two)")
!ollama pull llama3.2
print("Model ready!")

Starting Ollama server...
Pulling Llama 3.2 model... (This takes a minute or two)

Model ready!


In [ ]:
import subprocess
print("Starting server...")
subprocess.Popen(["ollama", "serve"])
print("Pulling Llama 3.1 (8B)... This will take a few minutes!")
!ollama pull llama3.1
print("Ready!")

Starting server...
Pulling Llama 3.1 (8B)... This will take a few minutes!

Ready!


In [ ]:
import subprocess
import time

print("Restarting Ollama server...")
# Boot up the background server
subprocess.Popen(["ollama", "serve"])
time.sleep(3) # Give it 3 seconds to fully wake up
print("Server is back online!")

Restarting Ollama server...
Server is back online!


In [ ]:
import os
import json
import requests
import gradio as gr
from faster_whisper import WhisperModel

# Configuration & Safety
OUTPUT_DIR = "/content/agent_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OLLAMA_URL = "http://localhost:11434/api/generate"

# Load Whisper on the Colab GPU for lightning-fast STT
print("Loading Whisper Model... (This takes a minute or two on the first run)")
stt_model = WhisperModel("small", device="cuda", compute_type="float16")

# 1. Tool Functions (The Hands)
def safe_create_file(filename, content=""):
    """Creates a file safely within the designated output folder."""
    safe_name = os.path.basename(filename)
    filepath = os.path.join(OUTPUT_DIR, safe_name)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)
    return f"Created {safe_name} in {OUTPUT_DIR}"

def summarize_text(text):
    """Uses LLM to summarize the given text."""
    payload = {
        "model": "llama3.1",
        "prompt": f"Please provide a highly detailed, comprehensive, and exhaustive summary or explanation of the following. Do not leave out any important details. If it is a complex topic, explain it fully in multiple paragraphs:\n\n{text}",
        "stream": False,
        "options": {
            "num_predict": 2500 # Maximize tokens to allow for exhaustive, long-form explanations
        }
    }
    response = requests.post(OLLAMA_URL, json=payload)
    return response.json().get("response", "Failed to summarize.")

def generate_code(filename, instructions):
    """Uses LLM to write full code without JSON constraints."""
    prompt = f"You are an expert software developer. Write the complete, fully functional code for a file named '{filename}' based on the following instructions.\n\nINSTRUCTIONS: {instructions}\n\nCRITICAL RULE: Output ONLY the raw code. Do NOT wrap the code in markdown blocks (no ```java or ```python). Do NOT include any explanations before or after the code. Just write the raw code directly."

    payload = {
        "model": "llama3.1",
        "prompt": prompt,
        "stream": False,
        "options": {
            "num_predict": 2000,
            "temperature": 0.2
        }
    }
    response = requests.post(OLLAMA_URL, json=payload)
    code = response.json().get("response", "").strip()

    # Failsafe Markdown Stripping
    if code.startswith("```"):
        code = code.split("\n", 1)[-1]
    if code.endswith("```"):
        code = code.rsplit("\n", 1)[0]

    return code.strip()

def general_chat_response(text, history):
    """Allows the LLM to chat freely without strict JSON constraints."""
    history_text = "\n".join([f"User: {h[0]}\nAgent: {h[1]}" for h in history[-3:]])
    prompt = f"Conversation History:\n{history_text}\n\nUser: {text}\nAgent:"

    payload = {
        "model": "llama3.1",
        "prompt": prompt,
        "stream": False,
        "options": {
            "num_predict": 800 # Gives the LLM plenty of room to explain complex topics like movies
        }
    }
    response = requests.post(OLLAMA_URL, json=payload)
    return response.json().get("response", "I'm having trouble thinking of a response.")

# 2. Core Pipeline (The Brain)
def transcribe(audio_path):
    """Converts audio file/mic to text."""
    if not audio_path:
        return "" # FIX: Return empty string instead of "No audio provided."
    # Force language to English to prevent phonetic alphabet hallucinations
    segments, _ = stt_model.transcribe(audio_path, beam_size=5, language="en")
    return " ".join([segment.text for segment in segments]).strip()

def get_intents(text, history):
    """Sends text and memory to LLM, returns a JSON array of commands."""
    history_text = "\n".join([f"User: {h[0]}\nAgent: {h[1]}" for h in history[-3:]])

    prompt = f"""
    You are a highly intelligent intent classification and execution engine.
    Analyze the user's command and extract EVERY distinct request into a list of actions.

    AVAILABLE INTENTS & SCHEMA:
    1. {{"intent": "write_code", "filename": "string", "instructions": "string"}}
    2. {{"intent": "summarize", "text_to_summarize": "string"}}
    3. {{"intent": "general_chat", "response": "string"}}

    CRITICAL RULES:
    - You MUST output a single JSON object containing an "actions" array: {{"actions": [...]}}
    - For 'write_code', NEVER write the actual code inside the JSON. Instead, provide detailed instructions for what the code should do in the 'instructions' field.
    - For 'summarize', if summarizing a file you just requested, set 'text_to_summarize' to EXACTLY the word "CODE_CONTEXT".
    - For 'general_chat', write the ACTUAL chat reply or joke in 'response'.
    - Combine file creation and coding into a single 'write_code' intent. Do NOT output multiple intents for creating the same file.
    - Identify EVERY action requested by the user. If they ask for 3 things, there MUST be 3 objects in the "actions" array.

    HISTORY:
    {history_text}

    COMMAND: "{text}"
    """

    try:
        response = requests.post(OLLAMA_URL, json={
            "model": "llama3.1",
            "prompt": prompt,
            "format": "json",
            "stream": False,
            "options": {
                "num_predict": 1500, # Increased further to allow for writing large scripts in JSON
                "temperature": 0.1
            }
        })

        raw_text = response.json().get("response", "{}").strip()

        # Sanitize Markdown just in case the LLM tries to be "helpful"
        if raw_text.startswith("```json"):
            raw_text = raw_text[7:]
        elif raw_text.startswith("```"):
            raw_text = raw_text[3:]

        if raw_text.endswith("```"):
            raw_text = raw_text[:-3]

        parsed = json.loads(raw_text.strip())

        # Extract the actions array from the root object
        if isinstance(parsed, dict) and "actions" in parsed:
            return parsed["actions"]
        elif isinstance(parsed, list):
            return parsed
        elif isinstance(parsed, dict):
            return [parsed]
        else:
            return [{"intent": "general_chat", "response": "I couldn't parse the intent."}]

    except Exception as e:
        print(f"DEBUG Error parsing JSON: {e}")
        return [{"intent": "general_chat", "response": "I didn't quite catch that format. Could you try again?"}]

def execute_agent(audio_file, chat_history, progress=gr.Progress()):
    """The main brain loop with progress tracking."""

    # Step 1: STT
    progress(0.1, desc="🎙️ Transcribing audio...")
    transcription = transcribe(audio_file)

    # Catch the empty string properly so it doesn't process empty input as a command
    if not transcription or transcription.strip() == "":
        return chat_history, "No audio detected", "N/A", "Please ensure your microphone is working and try again."

    # Step 2: Intent
    progress(0.4, desc="🧠 Analyzing intent...")
    actions = get_intents(transcription, chat_history)

    # Force single objects into a list to prevent the '.get' AttributeError
    if isinstance(actions, dict):
        actions = [actions]
    elif not isinstance(actions, list):
        actions = [{"intent": "general_chat", "response": "I couldn't parse the intent."}]

    # Step 3: Execution
    progress(0.7, desc="⚙️ Executing tools...")
    results = []
    agent_responses = []

    generated_code_context = "" # Save the code in memory to help the summarizer if needed

    for action in actions:
        intent = action.get("intent")

        if intent in ["create_file", "write_code"]:
            filename = action.get("filename", "unknown.txt")

            # Decoupled generation: Use the LLM to write the code outside of strict JSON limits!
            instructions = action.get("instructions", "")
            if not instructions: # Fallback
                 instructions = action.get("content", action.get("code_content", f"Write the script for {filename}"))

            progress(0.6, desc=f"💻 Writing code for {filename}...")
            content = generate_code(filename, instructions)

            res = safe_create_file(filename, content)

            generated_code_context += f"File {filename}:\n{content}\n"

            results.append(res)
            agent_responses.append(f"I created the file {filename}.")

        elif intent == "summarize":
            text_to_sum = action.get("text_to_summarize", transcription)

            # Catch the specific CODE_CONTEXT flag, or a lazy string
            if (text_to_sum.strip() == "CODE_CONTEXT" or len(text_to_sum) < 30) and generated_code_context:
                text_to_sum = generated_code_context
            elif not text_to_sum or text_to_sum.strip() == "CODE_CONTEXT":
                text_to_sum = transcription

            progress(0.8, desc="📝 Summarizing...")
            res = summarize_text(text_to_sum)
            results.append(f"Summary: {res}")
            agent_responses.append(f"Here is the summary: {res}")

        else:
            # general_chat
            resp = action.get("response", "")

            # If the intent parser was lazy (e.g. "quick joke"), use the dedicated free-speaking tool
            if not resp or len(resp) < 20 or resp.lower() in ["quick joke", "tell a joke", "chat"]:
                progress(0.8, desc="💬 Thinking of a response...")
                resp = general_chat_response(transcription, chat_history)

            results.append("Chatting")
            agent_responses.append(resp)

    # Step 4: Finalizing
    progress(0.9, desc="✨ Updating UI...")
    final_result_text = "\n".join(results)
    final_agent_speech = " ".join(agent_responses)

    chat_history.append((transcription, final_agent_speech))

    # Use ensure_ascii=False so that non-English characters render cleanly in the UI instead of as \uXXXX
    intent_str = json.dumps(actions, indent=2, ensure_ascii=False)

    progress(1.0, desc="Done!")
    return chat_history, transcription, intent_str, final_result_text

# 3. Gradio User Interface
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎙️ Voice-Controlled Local AI Agent")
    gr.Markdown("Speak a command like: *'Create a python file called math.py that adds two numbers, and also summarize what it does.'*")

    # Memory State
    history_state = gr.State([])

    with gr.Row():
        with gr.Column():
            audio_input = gr.Audio(type="filepath", label="Microphone Input")
            submit_btn = gr.Button("Execute Command", variant="primary")

        with gr.Column():
            chatbot = gr.Chatbot(label="Conversation History")

    with gr.Row():
        transcription_out = gr.Textbox(label="1. STT Transcription")
        intent_out = gr.Code(label="2. Detected Intent (JSON)", language="json")
        result_out = gr.Textbox(label="3. Execution Result")

    submit_btn.click(
        fn=execute_agent,
        inputs=[audio_input, history_state],
        outputs=[chatbot, transcription_out, intent_out, result_out]
    )

if __name__ == "__main__":
    demo.queue().launch(share=True, debug=True)

Loading Whisper Model... (This takes a minute or two on the first run)


/tmp/ipykernel_8992/2296152444.py:242: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_8992/2296152444.py:255: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Conversation History")
/tmp/ipykernel_8992/2296152444.py:255: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Conversation History")


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9687b1df625fe84c90.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://9687b1df625fe84c90.gradio.live
